# Module 9: Neural NLP for real messages

Build a training-only vocabulary and neural text classifier on human-labeled messages, and compare it with a TF-IDF/logistic baseline.

Use the complete source folder and the documented isolated CPU framework environment. Run in order, retain named text/tables, save, restart, and run all. First source/model downloads need internet; valid caches support replay. No earlier notebook state or paid account is needed.

The public script calls the same implementation. Small-model mechanics and valid software execution do not establish task quality or accessible notebook-interface operation.


In [ ]:
from pathlib import Path
import sys
import json
import tempfile
import platform
import importlib.metadata
import numpy as np
import pandas as pd

# Find shared code from the complete checkout/archive, not copied notebook logic.
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "ml_ai_it"]
ROOT = next((p for p in candidates if (p / "src/practical_ml_ai").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete ml_ai_it source folder.")
sys.path.insert(0, str(ROOT / "src"))
from practical_ml_ai.course_runner import run_module

# A new directory allows clean replay without silently replacing frozen evidence.
(ROOT / "runs").mkdir(exist_ok=True)
parent = Path(tempfile.mkdtemp(prefix="module09-notebook-", dir=ROOT / "runs"))
run_dir = parent / "study"
print("Evidence will be saved to:", run_dir)


## Inspect the real source and boundaries

Use all 5,159 deduplicated real SMS records with a seeded 60/20/20 split. Fit the vocabulary on training only (6,000-token cap, length 96, lowercase with punctuation retained). Preserving punctuation prevents an emoticon-only SMS from becoming all padding. Train an embedding/pooling classifier for at most 25 epochs with patience 5. Compare a class-balanced TF-IDF/logistic baseline; select each decision threshold on validation.

Before running, predict the kinds of errors and the relevant denominators. The source data and human labels remain unchanged.

In [ ]:
from practical_ml_ai.continuation_data import sms_splits
split, source_manifest = sms_splits(limit=None)
for name, part in split.items():
    # Display IDs/counts rather than raw messages that may contain personal data.
    print(name, "rows:", len(part), "positive labels:", int(part.label.sum()))
print("First training text IDs:", split["train"].text_id.head(3).tolist())
print("Sampling rule:", source_manifest["sampling"])


## Predict and inspect the mechanism

Vocabulary adaptation is fitting. Including final-test words in that step would let the holdout influence the representation. The label is a human spam/ham annotation, not a property inferred from a phone number alone. Class imbalance can make accuracy look good while recall is poor.

Tokens are discrete representations; embeddings are learned numeric vectors. Padding supports fixed-length batches and should not become a real word signal. Global average pooling summarizes token vectors. Unknown words and truncated messages limit what the model can use.


In [ ]:
# Whitespace counts are a readability diagnostic, not the fitted vocabulary.
word_counts = split["train"].text.str.split().str.len()
print("Training whitespace-word counts: min/median/max",
      int(word_counts.min()), float(word_counts.median()), int(word_counts.max()))
print("The actual neural vocabulary is adapted only inside the training workflow.")


## Guided construction before independent practice

A token ID is an index, not an amount. In this hand example 0 is padding, 1 and 2 index vectors [1,0] and [0,2]. The masked mean of [1,2,0] is [.5,1], not [1/3,2/3]. In the real model, training learns vectors; the hand table only explains the operation.

Predict the output before running. Then complete the module guided page's independent modification.


In [ ]:
import numpy as np
learning_ids = np.array([1,2,0])
learning_embedding = np.array([[0.,0.],[1.,0.],[0.,2.]])
learning_mask = learning_ids != 0
learning_pooled = learning_embedding[learning_ids][learning_mask].mean(axis=0)
print("vectors",learning_embedding[learning_ids],"pooled",learning_pooled)
assert np.allclose(learning_pooled,[.5,1.])

## Run the shared workflow

Predict how changing a threshold trades false positives against missed spam. The workflow chooses thresholds on validation, then evaluates the frozen decisions once on the final partition.

In [ ]:
# Execute the declared default protocol; final labels do not select settings.
result = run_module(9, run_dir)
print((run_dir / "summary.txt").read_text(encoding="utf-8"))

## Inspect the exact tables

Read neural and lexical confusion counts, precision, recall, ROC-AUC and average precision. Inspect validation_thresholds.csv to see why each threshold was selected. A result on this historical corpus does not establish performance on current messages. Compare the archived under-trained run only after interpreting your own outputs.

In [ ]:
table = pd.read_csv(run_dir / "learning_history.csv")
print("Evidence table: learning_history.csv")
print(table.head(15).to_string(index=False))
print("Complete report:", run_dir / "index.html")


## Verify one interpretation

Use the actual output to check a denominator, fitting boundary, update, or budget. A passing software check is separate from whether the method is useful.


In [ ]:
predictions = pd.read_csv(run_dir / "test_predictions.csv")
print("Actual positive labels:", int(predictions.actual.sum()))
print("Predicted positives:", int(predictions.predicted.sum()))
print("Precision/recall may be poor even if overall accuracy looks high.")
print("Training-only vocabulary:", result["vocabulary_fitted_on"])


## Record versions and save

Retain the source/protocol manifest, output directory, commands, versions, and your own interpretation. Complete the lesson's practice and reflection; optional depth is skippable. Restart-and-run-all makes hidden state visible, but repeated public final data are not fresh independent evaluation.


In [ ]:
versions = {"python": platform.python_version()}
for package in ["numpy","pandas","scikit-learn","tensorflow-cpu"]:
    versions[package] = importlib.metadata.version(package)
(run_dir / "environment.json").write_text(json.dumps(versions, indent=2) + "\n", encoding="utf-8")
print("Reproduction versions:", versions)
print("Your saved evidence:", run_dir)


## Independent practice

Audit the text-hash boundaries and vocabulary fitting scope. Compare neural and lexical baseline precision/recall and identify a limitation from length/vocabulary. Inspect raw messages locally only as needed; report IDs and aggregate findings rather than publishing personal contact strings.

Optional: Study a CNN/recurrent encoder, a larger vocabulary, or a length change with a predefined development protocol. Measure compute and errors. A different encoder is optional, not an entrance requirement.

Explain one limitation and a next action. If using GenAI, verify its suggestions against actual code/tables and follow local disclosure rules. Instructors can adapt the linked practice criteria while preserving learning goals and accessibility.
